# Comparacion de Notaciones: Big O, Omega y Theta

**Modulo:** Complexity Analysis  
**Objetivo:** Entender las diferencias entre las tres notaciones asintoticas  
**Duracion estimada:** 35 minutos

---

## Contenido

1. [Setup](#setup)
2. [Las Tres Notaciones](#las-tres-notaciones)
3. [Definiciones Formales](#definiciones-formales)
4. [Analisis por Caso](#analisis-por-caso)
5. [Ejemplos Comparativos](#ejemplos-comparativos)
6. [Ejercicios](#ejercicios)

---

## 1. Setup

In [ ]:
# Agregar path del proyecto
import sys
sys.path.insert(0, '../..')

# Imports necesarios
from app.core.parser import parse_pseudocode
from app.core.analyzer.complexity.big_o_analyzer import BigOAnalyzer
from app.core.analyzer.complexity.omega_analyzer import OmegaAnalyzer
from app.core.analyzer.complexity.complexity_calculator import ComplexityCalculator

# Para visualizacion
import pandas as pd

print("Setup completado")

---

## 2. Las Tres Notaciones

| Notacion | Nombre | Descripcion |
|----------|--------|-------------|
| O (Big O) | Cota Superior | Peor caso, limite maximo |
| Omega | Cota Inferior | Mejor caso, limite minimo |
| Theta | Cota Ajustada | Caso promedio, exacta |

### Ejemplo: Comparar Big O y Omega

In [ ]:
# Comparar Big O y Omega para busqueda lineal
codigo_busqueda = """
algorithm linearSearch(A[], n, key)
begin
    for i <- 1 to n do
        if (A[i] = key) then
            return i
        end
    end
    return -1
end
"""

ast = parse_pseudocode(codigo_busqueda)
big_o = BigOAnalyzer()
omega = OmegaAnalyzer()

resultado_o = big_o.analyze(ast)
resultado_omega = omega.analyze(ast)

print("=== BUSQUEDA LINEAL ===")
print(f"Big O (peor caso):   O({resultado_o})")
print(f"Omega (mejor caso):  Omega({resultado_omega})")
print("\nExplicacion:")
print("- Peor caso: el elemento esta al final o no existe -> O(n)")
print("- Mejor caso: el elemento esta al inicio -> Omega(1)")

---

## 3. Definiciones Formales

### Big O (Cota Superior)

```
f(n) = O(g(n)) si f(n) <= c * g(n) para n >= n0
```

### Omega (Cota Inferior)

```
f(n) = Omega(g(n)) si f(n) >= c * g(n) para n >= n0
```

### Theta (Cota Ajustada)

```
f(n) = Theta(g(n)) si c1*g(n) <= f(n) <= c2*g(n)
```

### Equivalencia

Theta(g(n)) existe cuando O(g(n)) = Omega(g(n))

### Ejemplo: Algoritmo con Theta

In [ ]:
# Selection Sort: tiene Theta porque siempre hace n^2 comparaciones
codigo_selection = """
algorithm selectionSort(A[], n)
begin
    for i <- 1 to n - 1 do
        min <- i
        for j <- i + 1 to n do
            if (A[j] < A[min]) then
                min <- j
            end
        end
        temp <- A[i]
        A[i] <- A[min]
        A[min] <- temp
    end
end
"""

ast = parse_pseudocode(codigo_selection)
calculator = ComplexityCalculator()
results = calculator.analyze_all(ast)

print("=== SELECTION SORT ===")
print(f"Big O:  O({results['big_o']})")
print(f"Omega:  Omega({results['omega']})")
print(f"Theta:  Theta({results['theta']})")
print("\nExplicacion:")
print("Selection Sort SIEMPRE hace n(n-1)/2 comparaciones,")
print("sin importar el orden inicial de los datos.")

---

## 4. Analisis por Caso

### Quick Sort - Ejemplo Clasico

| Caso | Complejidad | Cuando Ocurre |
|------|-------------|---------------|
| Mejor | n log n | Pivote divide en mitades |
| Peor | n^2 | Array ya ordenado |

### Ejemplo: Comparar Multiples Algoritmos

In [ ]:
# Comparar varios algoritmos
algoritmos = {
    "Bubble Sort": """
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
""",
    "Linear Search": """
algorithm linearSearch(A[], n, key)
begin
    for i <- 1 to n do
        if (A[i] = key) then
            return i
        end
    end
    return -1
end
""",
    "Sum Array": """
algorithm sumArray(A[], n)
begin
    sum <- 0
    for i <- 1 to n do
        sum <- sum + A[i]
    end
    return sum
end
"""
}

resultados = []
for nombre, codigo in algoritmos.items():
    ast = parse_pseudocode(codigo)
    results = calculator.analyze_all(ast)
    resultados.append({
        "Algoritmo": nombre,
        "Big O": f"O({results['big_o']})",
        "Omega": f"Omega({results['omega']})",
        "Theta": f"Theta({results['theta']})" if results['theta'] != 'N/A' else "N/A"
    })

df = pd.DataFrame(resultados)
print("=== COMPARACION DE ALGORITMOS ===")
print(df.to_string(index=False))

---

## 5. Cotas Ajustadas

### Cuando existe Theta?

Un algoritmo tiene Theta cuando:
- Big O = Omega (mismo comportamiento en todos los casos)

### Tabla de Algoritmos de Ordenamiento

| Algoritmo | Big O | Omega | Tiene Theta? |
|-----------|-------|-------|--------------|
| Bubble Sort | n^2 | n | No |
| Selection Sort | n^2 | n^2 | Si |
| Merge Sort | n log n | n log n | Si |

### Ejemplo: Analisis de Cotas

In [ ]:
# Funcion para determinar si existe cota ajustada
def tiene_theta(big_o, omega):
    """Determina si un algoritmo tiene cota ajustada"""
    return str(big_o) == str(omega)

# Analizar varios algoritmos
algoritmos_theta = [
    ("Linear Search", codigo_busqueda),
    ("Selection Sort", codigo_selection),
    ("Sum Array", algoritmos["Sum Array"])
]

print("=== ANALISIS DE COTA AJUSTADA ===")
for nombre, codigo in algoritmos_theta:
    ast = parse_pseudocode(codigo)
    results = calculator.analyze_all(ast)
    
    o = results['big_o']
    omega = results['omega']
    theta_existe = tiene_theta(o, omega)
    
    print(f"\n{nombre}:")
    print(f"  Big O: O({o}), Omega: Omega({omega})")
    print(f"  Tiene Theta: {'Si -> Theta(' + str(o) + ')' if theta_existe else 'No'}")

---

## 6. Ejercicios

### Ejercicio 1: Predice las Complejidades

In [ ]:
# Ejercicio 1: Que complejidades tiene este algoritmo?
# Primero predice, luego ejecuta para verificar

codigo_ejercicio = """
algorithm findMax(A[], n)
begin
    max <- A[1]
    for i <- 2 to n do
        if (A[i] > max) then
            max <- A[i]
        end
    end
    return max
end
"""

# Tu prediccion:
# Big O: ___
# Omega: ___
# Theta: ___

# Verificar
ast = parse_pseudocode(codigo_ejercicio)
results = calculator.analyze_all(ast)
print("Resultado:")
print(f"Big O: O({results['big_o']})")
print(f"Omega: Omega({results['omega']})")
print(f"Theta: {results['theta']}")

### Ejercicio 2: Crear Algoritmo con Theta

In [ ]:
# Ejercicio 2: Escribe un algoritmo que tenga Theta(n^2)
# Pista: debe tener el mismo comportamiento en todos los casos

tu_codigo = """
algorithm miAlgoritmo(A[], n)
begin
    for i <- 1 to n do
        for j <- 1 to n do
            x <- A[i] + A[j]
        end
    end
end
"""

ast = parse_pseudocode(tu_codigo)
results = calculator.analyze_all(ast)
print(f"Big O: O({results['big_o']})")
print(f"Omega: Omega({results['omega']})")
print(f"Tiene Theta: {results['big_o'] == results['omega']}")

---

## 7. Tips y Resumen

### Tips

- **Big O** para garantias de rendimiento (nunca sera peor que...)
- **Omega** para limites teoricos (al menos tomara...)
- **Theta** existe cuando mejor = peor caso
- Usa `ComplexityCalculator` para analisis completo

### Resumen

Has aprendido:
- Las diferencias entre Big O, Omega y Theta
- Cuando existe una cota ajustada
- Como comparar algoritmos usando las tres notaciones
- Ejemplos practicos de analisis

---

## Proximos Pasos

- **recurrence_solver.ipynb**: Resolver ecuaciones de recurrencia
- **big_o_examples.ipynb**: Mas ejemplos de Big O

**Siguiente modulo recomendado**: `03_pattern_detection/` para detectar patrones